# TrOCR Fine-Tuning - Kaggle (T4 x2) or your own computer

Fine-tunes Microsoft's `microsoft/trocr-base-handwritten` on your own dataset, on **Kaggle** or on **your own
computer's GPU** (tested on an RTX 4050 Laptop GPU, 6 GB). Cell 4 sees where it runs, picks the training settings
from the dataset size (profiles 5k, 10k, 15k, 20k, 30k, 50k, 100k) and fits them to the GPU.
To continue training a model you already fine-tuned, use `trocr-finetuning-from-model.ipynb` instead.

**Run on Kaggle**
1. Settings -> Accelerator -> **GPU T4 x2**, and Settings -> Internet -> **On** (to download the base model).
2. Add Input -> your dataset: `manifest.csv` (columns `filename,label,split`) + `train/ val/ test/` folders.
   Add only that one dataset; cell 4 finds it.
3. In cell 4, set `CRMS_MODEL_KEY` - the name you will upload the model as. Blank = `TrOcr-100k-samples`
   (the number follows your dataset size).
4. Run it with **Save Version -> Save & Run All (Commit)**, so it keeps running if your browser disconnects.
   Expect about **3 hours for 100k samples** (Kaggle allows 12 hours per run).

**Run on your own computer**
1. Open this notebook in VS Code (or Jupyter) with a Python that has a CUDA build of PyTorch, `transformers`,
   `pandas`, `Pillow`, `tqdm` and `ipykernel`. The first run downloads the base model (about 1.3 GB) unless it is
   already in the Hugging Face cache.
2. In cell 4, set `LOCAL_DATA_ROOT` to your dataset folder (relative to this notebook, or a full path), and
   `CRMS_MODEL_KEY`.
3. Run All. Cell 4 prints a rough time estimate. Each run is saved in its own folder under `LOCAL_RUNS_DIR`
   (default: `trocr-runs` in your user folder - outside OneDrive, so the 1.3 GB model and `test-errors.csv`
   are not uploaded anywhere). A 6 GB GPU trains with the Adafactor optimizer, because AdamW needs about 10 GB;
   expect about 30-35 minutes for 12k samples. Keep a laptop plugged in.

**What this notebook does**
- Sets a random seed, so running it again on the same data gives the same numbers.
- Checks the data before training: missing images, exact copies of test images in training or validation
  (left out), test texts that also appear in training (counted and scored apart), labels that are too long.
- Varies the training pictures slightly (tilt, blur, contrast, ink thickness, width) so the model copes better
  with real scans. Validation and test pictures are never changed. `AUGMENT = False` in cell 4 turns it off.
- Adds CRMS's 8 px paper-coloured border to every picture (training, validation and test), because CRMS reads its
  crops with it. `CRMS_BORDER_PX = 0` in cell 4 turns it off, for crops exported by `crms:export-training`, which
  already have it.
- Scores Microsoft's base model and your fine-tuned model on the **same locked test set**, the way CRMS reads
  (greedy, at most 32 tokens per crop, in fp16 like CRMS's default), and keeps the best epoch by validation CER -
  only if it beats the base model.
- Training on synthetic data? Set `VAL_DATA_ROOT` in cell 4 to your real dataset: the best epoch is then picked on
  real validation pictures (never its test pictures), because synthetic validation is too easy to tell epochs apart.

**What you get** (Kaggle: the Output panel; your own computer: the run folder): `trocr-finetuned.zip` - upload it
in the CRMS OCR Workspace. It holds the model, `evaluation-report.json` (its test scores),
`base-evaluation-report.json` (the base model on the same test, so the workspace compares like with like) and
`training-summary.json` (every setting and number of this run).
Also: `test-errors.csv` (every test image either model read wrong) and `augmentation-preview.png`.

**Real ledger data** is civil registry data. Train on it **on your own computer**, so it never leaves it; do not
upload it to Kaggle unless you are allowed to (and then only as a private dataset). To score a model on CRMS's own
real line crops, run `php artisan crms:export-training --test-set=real-lines` and then
`python ml\test_finetuned.py --model <model> --dataset real-lines` (see docs/DEVELOPER-GUIDE.md).

Scores can differ slightly from the 50k run made with the older version of this notebook: that version read up to
40 tokens per crop, did not vary the training pictures, and counted letter case (this one compares in capitals,
as CRMS shows its readings). Runs made before the CRMS border and fp16 reading were added are not directly
comparable with runs made after.

In [ ]:
# 1. (Optional) ensure a recent transformers. Kaggle usually has it preinstalled.
# Uncomment if you hit version issues.
# !pip install -q -U transformers

print('[OK] cell ready')

In [ ]:
# 2. Imports and small helpers
import os, io, gc, re, json, math, time, random, shutil, hashlib, tempfile, zipfile, warnings, logging
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

import numpy as np
import pandas as pd
# pandas 3 keeps text in pyarrow by default, which crashed on Windows once PyTorch had loaded first.
# Plain Python strings avoid that and change nothing else here.
pd.set_option('mode.string_storage', 'python')
import torch
import transformers
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import (TrOCRProcessor, VisionEncoderDecoderModel,
                          get_linear_schedule_with_warmup)
from transformers.optimization import Adafactor

WEIGHT_FILES = ('model.safetensors', 'pytorch_model.bin')


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as source:
        while chunk := source.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def weights_file_in(folder):
    return next((name for name in WEIGHT_FILES if os.path.isfile(os.path.join(folder, name))), None)


def find_datasets(root):
    """Folders under `root` that hold a dataset (a manifest.csv)."""
    found = []
    for here, dirs, files in os.walk(root):
        dirs.sort()
        if 'manifest.csv' in files:
            found.append(os.path.normpath(here))
            dirs[:] = [d for d in dirs if d not in ('train', 'val', 'test')]  # do not walk the images
    return found


print('[OK] Imports | torch', torch.__version__, '| transformers', transformers.__version__)

In [ ]:
# 3. GPU check
print('CUDA available :', torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print('GPU count      :', n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. On Kaggle: Settings -> Accelerator -> GPU T4 x2. On your own computer: run this '
                       'notebook with a Python that has a CUDA build of PyTorch (pytorch.org/get-started/locally).')
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1e9

In [ ]:
# 4. CONFIG - where it runs, names, dataset, and the training settings for the dataset size and the GPU
MODEL_NAME = 'microsoft/trocr-base-handwritten'
# The exact Hugging Face version of the base model. CRMS installs this same version
# in ml/models/base, so the base scores this run measures can be checked against it.
MODEL_REVISION = 'eaacaf452b06415df8f10bb6fad3a4c11e609406'

# ---- Where it runs ------------------------------------------------------------------------------------
RUN_ON = 'auto'   # 'auto' = Kaggle when /kaggle/input exists, otherwise this computer; or 'kaggle' / 'local'
KAGGLE_INPUT = '/kaggle/input'
KAGGLE_WORKING = '/kaggle/working'
# On your own computer: the dataset folder (relative to this notebook, or a full path), and where runs are
# saved, each in its own folder. Keep runs outside OneDrive: the model is 1.3 GB and test-errors.csv holds
# the test labels.
LOCAL_DATA_ROOT = r'datasets\raw-cropped-dataset'
LOCAL_RUNS_DIR = os.path.join(os.path.expanduser('~'), 'trocr-runs')

# Names written into the CRMS reports.
CRMS_MODEL_KEY = ''  # the name you will upload it as, e.g. 'TrOcr-100k-broken-samples'. Blank = 'TrOcr-<size>k-samples'
DATASET_LABEL = ''   # e.g. 'synthetic-broken-handwritten-samples100k'. Blank = the dataset's (folder) name
DATA_ROOT = ''       # blank = Kaggle: the one folder with manifest.csv under /kaggle/input; this computer: LOCAL_DATA_ROOT
# Optional: a second dataset whose VAL split picks the best epoch - e.g. your real dataset when this run trains on
# synthetic data, whose own validation is too easy to tell epochs apart (the 50k run reached 0.01% CER on it but
# reads real crops at 36%). Only its val rows are read, never its test rows. Blank = this dataset's own val split.
# On Kaggle: its full /kaggle/input/... path.
VAL_DATA_ROOT = ''   # e.g. r'datasets\raw-cropped-dataset'
MAX_TRAIN_SAMPLES = 0  # 0 = every training image; e.g. 20000 = a random 20k of them (validation and test stay whole)

# ---- Training settings by dataset size ------------------------------------------------------------------
# EFFECTIVE_BATCH = images per weight update. Every GPU setup below reaches it with its own images per step
# x gradient accumulation, so a 6 GB laptop trains with the same settings as Kaggle's two T4s.
PROFILES = {
    '5k': dict(EPOCHS=4, EFFECTIVE_BATCH=32,
               DECODER_LR=2e-5, ENCODER_LR=1e-5, WARMUP_RATIO=0.10, WEIGHT_DECAY=0.01,
               FREEZE_ENCODER=True, UNFREEZE_AT_EPOCH=1,
               EARLY_STOP_PATIENCE=2, MAX_LABEL_LENGTH=40),
    '10k': dict(EPOCHS=3, EFFECTIVE_BATCH=64,
                DECODER_LR=3e-5, ENCODER_LR=1.5e-5, WARMUP_RATIO=0.05, WEIGHT_DECAY=0.01,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=2, MAX_LABEL_LENGTH=40),
    # 15k: e.g. 12k real + 5k synthetic, or a 15k synthetic set. The 10k settings: at this size 3 epochs at
    # batch 64 give twice the weight updates of 2 epochs at batch 96.
    '15k': dict(EPOCHS=3, EFFECTIVE_BATCH=64,
                DECODER_LR=3e-5, ENCODER_LR=1.5e-5, WARMUP_RATIO=0.05, WEIGHT_DECAY=0.01,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=2, MAX_LABEL_LENGTH=40),
    '20k': dict(EPOCHS=2, EFFECTIVE_BATCH=96,
                DECODER_LR=3e-5, ENCODER_LR=1.5e-5, WARMUP_RATIO=0.05, WEIGHT_DECAY=0.01,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
    # 30k: between the 20k and 50k settings.
    '30k': dict(EPOCHS=2, EFFECTIVE_BATCH=96,
                DECODER_LR=2.5e-5, ENCODER_LR=1.25e-5, WARMUP_RATIO=0.04, WEIGHT_DECAY=0.01,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
    '50k': dict(EPOCHS=2, EFFECTIVE_BATCH=128,
                DECODER_LR=2e-5, ENCODER_LR=1e-5, WARMUP_RATIO=0.03, WEIGHT_DECAY=0.01,
                FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
    # 100k: the 50k settings, which reached 0.01% CER on synthetic validation in 2 epochs, over twice the samples
    # (about 625 updates of 128 images per epoch).
    '100k': dict(EPOCHS=2, EFFECTIVE_BATCH=128,
                 DECODER_LR=2e-5, ENCODER_LR=1e-5, WARMUP_RATIO=0.03, WEIGHT_DECAY=0.01,
                 FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
                 EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
}
PROFILE = 'auto'          # 'auto' = the profile closest to your dataset size, or '5k' ... '100k'

# ---- GPU setups ---------------------------------------------------------------------------------------
# BATCH = images per step (on Kaggle split across the 2 T4s by DataParallel).
# OPTIMIZER: 'adamw' keeps two extra copies of every weight (about 9.6 GB at batch 16); 'adafactor' keeps a
# small summary instead, so it fits a 6 GB laptop GPU. Adafactor can end slightly better or worse than AdamW,
# so compare runs on the same test set.
# CHECKPOINTING: recompute each layer's in-between results in the backward pass instead of keeping them.
# A 6 GB GPU needs it: with gradient accumulation the 1.3 GB of gradients stay in memory, and batch 8 without
# it reached 6.27 GB - past the card, so Windows moved memory to RAM and training ran 4x slower.
# TRAIN_MS / READ_MS (milliseconds per image) only feed the rough time estimate below: measured with this
# notebook on an RTX 4050 Laptop; for Kaggle worked out from this notebook's 100k run (about 3 hours).
GPU_SETUPS = {
    'kaggle':   dict(BATCH=16, OPTIMIZER='adamw', CHECKPOINTING=False, EVAL_BATCH_SIZE=32, TRAIN_MS=60, READ_MS=40),
    'gpu-11gb': dict(BATCH=16, OPTIMIZER='adamw', CHECKPOINTING=False, EVAL_BATCH_SIZE=32, TRAIN_MS=None, READ_MS=None),
    'gpu-6gb':  dict(BATCH=16, OPTIMIZER='adafactor', CHECKPOINTING=True, EVAL_BATCH_SIZE=16, TRAIN_MS=55, READ_MS=40),
}
GPU_SETUP = 'auto'        # 'auto' = 'kaggle' on Kaggle; on your computer 'gpu-11gb' from 11 GB of GPU memory, else 'gpu-6gb'

EVAL_MAX_NEW_TOKENS = 32  # CRMS reads at most 32 tokens per crop (ml/api/main.py), so the scores match the app
VAL_SAMPLES = 5000        # validation images scored after each epoch to pick the best one; 0 = all of them
SEED = 20261102
AUGMENT = True            # vary the training pictures slightly (cell 8); validation and test never change
CHECK_DUPLICATE_IMAGES = True  # leave exact copies of test images out of training and validation
# CRMS cuts every crop with an 8 px border of the paper's own colour around the writing
# (ml/line_markers.py CROP_PADDING). The same border is added to every picture here - training,
# validation and test - so the model learns from, and is scored on, what CRMS shows it.
# 0 = off: for a dataset exported by `php artisan crms:export-training`, whose crops already have it.
CRMS_BORDER_PX = 8
# How CRMS keeps the model when it reads: 'fp16' (its default, OCR_PRECISION in CRMS's .env) or 'fp32'.
# The test set is read this way, so the reports match the app. Validation after each epoch (only to
# pick the best epoch) reads in fp32.
READ_PRECISION = 'fp16'
seed_everything(SEED)
if READ_PRECISION not in ('fp16', 'fp32'):
    raise ValueError("READ_PRECISION must be 'fp16' or 'fp32'")
READ_DTYPE = torch.float16 if READ_PRECISION == 'fp16' else torch.float32

# ---- Find the dataset -----------------------------------------------------------------------------------
if RUN_ON not in ('auto', 'kaggle', 'local'):
    raise ValueError("RUN_ON must be 'auto', 'kaggle' or 'local'")
IS_KAGGLE = RUN_ON == 'kaggle' or (RUN_ON == 'auto' and os.path.isdir(KAGGLE_INPUT))
if VAL_DATA_ROOT:
    VAL_DATA_ROOT = os.path.normpath(VAL_DATA_ROOT) if IS_KAGGLE else os.path.abspath(VAL_DATA_ROOT)
if IS_KAGGLE:
    found_datasets = find_datasets(KAGGLE_INPUT)
    print('Datasets found (folders with manifest.csv):')
    for path in found_datasets or ['(none)']:
        print('   ', path)
    training_datasets = [path for path in found_datasets if path != VAL_DATA_ROOT]
    if DATA_ROOT:
        DATA_ROOT = os.path.normpath(DATA_ROOT)
    elif len(training_datasets) == 1:
        DATA_ROOT = training_datasets[0]
    else:
        raise RuntimeError('Set DATA_ROOT to your dataset folder (one of those listed above).' if found_datasets
                           else 'No manifest.csv under /kaggle/input. Add your dataset with Add Input.')
else:
    DATA_ROOT = os.path.abspath(DATA_ROOT or LOCAL_DATA_ROOT)
MANIFEST_CSV = os.path.join(DATA_ROOT, 'manifest.csv')
if not os.path.isfile(MANIFEST_CSV):
    raise FileNotFoundError(f'No manifest.csv in DATA_ROOT: {DATA_ROOT}'
                            + ('' if IS_KAGGLE else ' - set LOCAL_DATA_ROOT in this cell to your dataset folder.'))


def kaggle_dataset_name(root):
    """The dataset's Kaggle name from its path: /kaggle/input/<name>/... or /kaggle/input/datasets/<owner>/<name>/..."""
    parts = os.path.relpath(root, KAGGLE_INPUT).replace('\\', '/').split('/')
    if parts[0] in ('.', '..'):
        return os.path.basename(os.path.normpath(root))
    if parts[0] == 'datasets' and len(parts) >= 3:
        return parts[2]
    return parts[0]


def read_manifest(root):
    frame = pd.read_csv(os.path.join(root, 'manifest.csv'), keep_default_na=False)
    missing = {'filename', 'label', 'split'} - set(frame.columns)
    if missing:
        raise ValueError(f'{root}/manifest.csv has no {sorted(missing)} column')
    frame['filename'] = frame['filename'].astype(str).str.strip()
    frame['label'] = frame['label'].astype(str).str.strip()
    frame['split'] = frame['split'].astype(str).str.strip().str.lower()
    # Rows with no text, or marked UNREADABLE (in any case, with or without spaces), are not used.
    frame = frame[(frame['label'] != '') & (frame['label'].str.upper() != 'UNREADABLE')].copy()
    frame['path'] = [os.path.join(root, split, name) for split, name in zip(frame['split'], frame['filename'])]
    return frame.reset_index(drop=True)


manifest = read_manifest(DATA_ROOT)
val_manifest = None
if VAL_DATA_ROOT:
    if not os.path.isfile(os.path.join(VAL_DATA_ROOT, 'manifest.csv')):
        raise FileNotFoundError(f'No manifest.csv in VAL_DATA_ROOT: {VAL_DATA_ROOT}')
    if os.path.normcase(VAL_DATA_ROOT) == os.path.normcase(DATA_ROOT):
        VAL_DATA_ROOT = ''   # the same dataset: its own val split
    else:
        val_manifest = read_manifest(VAL_DATA_ROOT)
        if not (val_manifest['split'] == 'val').any():
            raise ValueError(f'VAL_DATA_ROOT has no usable val rows: {VAL_DATA_ROOT}')
n_samples = len(manifest)
n_train_rows = int((manifest['split'] == 'train').sum())
n_train_used = min(n_train_rows, MAX_TRAIN_SAMPLES) if MAX_TRAIN_SAMPLES > 0 else n_train_rows
n_used = n_samples - n_train_rows + n_train_used   # the dataset size the profile is picked for
PROFILE_SIZES = {name: int(name[:-1]) * 1000 for name in PROFILES}
picked = min(PROFILE_SIZES, key=lambda name: abs(math.log(max(n_used, 1) / PROFILE_SIZES[name])))
if PROFILE == 'auto':
    PROFILE = picked
elif PROFILE != picked:
    print(f'WARNING: PROFILE {PROFILE!r} is set, but {n_used:,} samples is closest to {picked!r}.')

# ---- The GPU setup, and the settings this run trains with -------------------------------------------------
if GPU_SETUP == 'auto':
    GPU_SETUP = 'kaggle' if IS_KAGGLE else ('gpu-11gb' if GPU_MEMORY_GB >= 11 else 'gpu-6gb')
setup = GPU_SETUPS[GPU_SETUP]
batch = min(setup['BATCH'], PROFILES[PROFILE]['EFFECTIVE_BATCH'])
cfg = {**PROFILES[PROFILE], 'BATCH_SIZE': batch,
       'GRAD_ACCUM_STEPS': max(1, round(PROFILES[PROFILE]['EFFECTIVE_BATCH'] / batch)),
       'OPTIMIZER': setup['OPTIMIZER'], 'GRADIENT_CHECKPOINTING': setup['CHECKPOINTING']}
EVAL_BATCH_SIZE = setup['EVAL_BATCH_SIZE']  # crops per generate() call when scoring (on 1 GPU)
# Loader workers: none on Windows, where worker processes cannot load the dataset class defined in this notebook.
NUM_WORKERS = 0 if os.name == 'nt' else min(4, os.cpu_count() or 2)

size_name = f'{round(n_used / 1000)}k' if n_used >= 1000 else str(n_used)
CRMS_MODEL_KEY = CRMS_MODEL_KEY.strip() or f'TrOcr-{size_name}-samples'
DATASET_LABEL = DATASET_LABEL.strip() or (kaggle_dataset_name(DATA_ROOT) if IS_KAGGLE
                                          else os.path.basename(os.path.normpath(DATA_ROOT)))
if IS_KAGGLE:
    WORKING_DIR = KAGGLE_WORKING
else:
    # Each run in its own folder, so a second run never deletes the first one's model.
    run_name = re.sub(r'[^A-Za-z0-9._-]+', '-', CRMS_MODEL_KEY) + datetime.now().strftime('_%Y%m%d-%H%M%S')
    WORKING_DIR = os.path.join(LOCAL_RUNS_DIR, run_name)
os.makedirs(WORKING_DIR, exist_ok=True)
OUTPUT_DIR = os.path.join(WORKING_DIR, 'trocr-finetuned')

n_val = int(((manifest if val_manifest is None else val_manifest)['split'] == 'val').sum())
n_val_scored = min(n_val, VAL_SAMPLES) if VAL_SAMPLES > 0 else n_val
n_test = int((manifest['split'] == 'test').sum())
print()
print('Runs on    :', 'Kaggle' if IS_KAGGLE else 'this computer', f'| GPU setup {GPU_SETUP!r}:',
      torch.cuda.get_device_name(0), f'({GPU_MEMORY_GB:.1f} GB)' + (f' x{n_gpu}' if n_gpu > 1 else ''))
print('DATA_ROOT  =', DATA_ROOT)
if VAL_DATA_ROOT:
    print('VAL_DATA_ROOT =', VAL_DATA_ROOT, '(its val split picks the best epoch)')
print(f'Samples    : {n_samples:,} usable rows ->', {s: int((manifest["split"] == s).sum()) for s in ("train", "val", "test")})
if n_train_used < n_train_rows:
    print(f'             training on a random {n_train_used:,} of the {n_train_rows:,} training images (MAX_TRAIN_SAMPLES)')
print(f'Profile    : {PROFILE}')
for k, v in cfg.items():
    print(f'  {k:20}: {v}')
print(f"  -> {cfg['BATCH_SIZE']} images per step x {cfg['GRAD_ACCUM_STEPS']} steps = "
      f"{cfg['BATCH_SIZE'] * cfg['GRAD_ACCUM_STEPS']} images per weight update")
print(f'Seed {SEED} | augment {AUGMENT} | CRMS border {CRMS_BORDER_PX} px | test read in {READ_PRECISION} | '
      f'reading limit {EVAL_MAX_NEW_TOKENS} tokens | {NUM_WORKERS} loader workers')
if setup['TRAIN_MS']:
    reads = 2 * n_test + n_val_scored * (1 + cfg['EPOCHS'])  # base test + val, val after each epoch, fine-tuned test
    minutes = (n_train_used * cfg['EPOCHS'] * setup['TRAIN_MS'] + reads * setup['READ_MS']) / 60000
    print(f'Rough time : about {minutes:.0f} min if every epoch runs (early stopping can end it sooner)')
print('Report names: model =', repr(CRMS_MODEL_KEY), '| dataset =', repr(DATASET_LABEL))
print('Saving to  :', WORKING_DIR)
print('>>> Upload the ZIP in CRMS under this exact model name:', repr(CRMS_MODEL_KEY))

In [ ]:
# 5. Metrics (dependency-free CER / WER / exact match) and a chance check
def _levenshtein(ref, hyp):
    m, n = len(ref), len(hyp)
    if m == 0:
        return n
    if n == 0:
        return m
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            curr[j] = min(prev[j]+1, curr[j-1]+1, prev[j-1]+cost)
        prev = curr
    return prev[n]


# CRMS shows every reading in capital letters, so readings and labels are compared in capitals:
# "maasin" and "Maasin" both count as right. Set to False to also score letter case.
SCORE_IN_CAPITALS = True


def as_crms(text):
    text = str(text)
    return text.upper() if SCORE_IN_CAPITALS else text


def compute_metrics(references, hypotheses):
    tce = tc = twe = tw = exact = 0
    for ref, hyp in zip(references, hypotheses):
        ref, hyp = as_crms(ref), as_crms(hyp)
        tce += _levenshtein(list(ref), list(hyp)); tc += len(ref)
        rw, hw = ref.split(), hyp.split()
        twe += _levenshtein(rw, hw); tw += len(rw)
        if ref == hyp:
            exact += 1
    n = len(references)
    return {'cer': (tce/tc) if tc else 0.0, 'wer': (twe/tw) if tw else 0.0,
            'accuracy': (exact/n) if n else 0.0, 'exact': exact, 'total': n}


def print_metrics(m, title='EVALUATION METRICS'):
    print('='*56); print(title); print('='*56)
    print(f"  Samples evaluated : {m['total']}")
    print(f"  Exact match       : {m['exact']}/{m['total']} ({m['accuracy']*100:.2f}%)")
    print(f"  CER               : {m['cer']*100:.2f}%")
    print(f"  WER               : {m['wer']*100:.2f}%")
    if 'ms_per_image' in m:
        print(f"  Reading time      : {m['ms_per_image']:.1f} ms per image (batches of {EVAL_BATCH_SIZE}, 1 GPU)")
    print('='*56)


def mcnemar_p(fixed, broke):
    """Exact two-sided McNemar test on the images only one of two models read right.
    Below 0.05, the difference between the two models is unlikely to be chance."""
    n = fixed + broke
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(fixed, broke) + 1)) / 2 ** n
    return min(1.0, 2 * tail)


print('[OK] Metrics')

In [ ]:
# 6. Load the base processor + model and apply the decoder-start fix
def load_base_processor():
    try:
        return TrOCRProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    except ValueError:
        # Newer transformers 5.x no longer builds this model's tokenizer from vocab.json + merges.txt.
        # Microsoft's repository has the missing tokenizer.json in an open pull request (refs/pr/11);
        # CRMS's ml/download_trocr.py uses the same file for ml/models/base.
        folder = tempfile.mkdtemp(prefix='base-processor-')  # outside /kaggle/working, so not in Output
        for name in ('preprocessor_config.json', 'tokenizer_config.json', 'special_tokens_map.json',
                     'vocab.json', 'merges.txt'):
            shutil.copy(hf_hub_download(MODEL_NAME, name, revision=MODEL_REVISION), folder)
        shutil.copy(hf_hub_download(MODEL_NAME, 'tokenizer.json', revision='refs/pr/11'), folder)
        print('Tokenizer: added tokenizer.json from refs/pr/11 (this transformers version needs it)')
        return TrOCRProcessor.from_pretrained(folder)


processor = load_base_processor()
tokenizer = processor.tokenizer


def prepare_for_reading(model, proc):
    """TrOCR is pretrained to start decoding from the sep/eos token (id 2), not cls (id 0).
    Keep config and generation_config in step, so training and reading agree.
    use_cache makes reading about twice as fast; CRMS turns it on too (ml/api/main.py)."""
    tok = proc.tokenizer
    model.config.decoder_start_token_id = tok.sep_token_id
    model.config.eos_token_id = tok.sep_token_id
    model.config.pad_token_id = tok.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    gen = model.generation_config
    gen.decoder_start_token_id = model.config.decoder_start_token_id
    gen.eos_token_id = model.config.eos_token_id
    gen.pad_token_id = model.config.pad_token_id
    gen.max_new_tokens = EVAL_MAX_NEW_TOKENS
    gen.max_length = None  # max_new_tokens is the limit
    gen.use_cache = True
    return model


def load_for_test(folder, **kwargs):
    """A separate copy of a model, kept the way CRMS keeps it for reading (READ_PRECISION: loaded
    in fp32, then turned to fp16, as ml/api/main.py does). Only for scoring the test set."""
    model = VisionEncoderDecoderModel.from_pretrained(folder, **kwargs)
    if READ_DTYPE != torch.float32:
        model.to(READ_DTYPE)
    return prepare_for_reading(model, processor).to(device).eval()


raw_model = prepare_for_reading(
    VisionEncoderDecoderModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION), processor).to(device)
print('Decoder start token id:', raw_model.config.decoder_start_token_id, '(should be 2)')
print('Params:', f'{sum(p.numel() for p in raw_model.parameters()):,}')

In [ ]:
# 7. DATA - splits, missing images, copies of test images, test texts seen in training, label lengths
def split_of(frame, split):
    return frame[frame['split'] == split].reset_index(drop=True)


def md5_of(path):
    with open(path, 'rb') as source:
        return hashlib.md5(source.read()).hexdigest()


train_all, val_all, test_df = (split_of(manifest, s) for s in ('train', 'val', 'test'))
if val_manifest is not None:
    val_all = split_of(val_manifest, 'val')  # the best epoch is picked on VAL_DATA_ROOT's validation pictures
for name, frame in (('train', train_all), ('val', val_all), ('test', test_df)):
    if frame.empty:
        raise ValueError(f'The dataset has no usable {name} rows in manifest.csv.')

# Every image must exist before training starts, not fail an hour into an epoch.
listings = {}


def listed(folder):
    if folder not in listings:
        listings[folder] = set(os.listdir(folder)) if os.path.isdir(folder) else set()
    return listings[folder]


missing_files = [path for frame in (train_all, val_all, test_df) for path in frame['path']
                 if os.path.basename(path) not in listed(os.path.dirname(path)) and not os.path.isfile(path)]
if missing_files:
    raise FileNotFoundError(f'{len(missing_files):,} images named in manifest.csv are missing, e.g. {missing_files[:3]}')

# Exact copies (the same file bytes) of test images are never trained or validated on,
# and validation images are not copies of training images.
frames = {'train': train_all, 'val': val_all, 'test': test_df}
if CHECK_DUPLICATE_IMAGES:
    with ThreadPoolExecutor(max_workers=8) as pool:
        for name, frame in frames.items():
            frame['md5'] = list(tqdm(pool.map(md5_of, frame['path']), total=len(frame),
                                     desc=f'Checking {name} images', leave=False))
else:
    for frame in frames.values():
        frame['md5'] = frame['path']  # without the check, only the very same file counts as a copy
test_md5 = set(test_df['md5'])
train_unique = train_all[~train_all['md5'].isin(test_md5)].reset_index(drop=True)
train_df = (train_unique.sample(n=MAX_TRAIN_SAMPLES, random_state=SEED)  # a random part of them (cell 4)
            if 0 < MAX_TRAIN_SAMPLES < len(train_unique) else train_unique).reset_index(drop=True)
val_usable = val_all[~val_all['md5'].isin(test_md5 | set(train_df['md5']))]
val_df = (val_usable.sample(n=VAL_SAMPLES, random_state=SEED) if 0 < VAL_SAMPLES < len(val_usable)
          else val_usable).reset_index(drop=True)
if val_df.empty:
    raise ValueError('No validation images left after removing copies of test and training images.')

# Test texts that also appear in training. A test image of a seen text checks reading a new
# picture of a known word; the unseen ones check reading words the model never trained on.
# Compared in capitals, like the scores: "Maasin" in training and "MAASIN" in test are the same text.
seen_texts = {as_crms(text) for text in train_df['label']}
test_df['label_seen'] = [as_crms(text) in seen_texts for text in test_df['label']]

# Pages: the crops of one page share its clerk, ink and scan. If one page has crops in training and in
# test, the test is easier than a new ledger book will be. Checked when manifest.csv names each crop's page.
PAGE_COLUMNS = ('page_id', 'page', 'source_page', 'document_id')
page_column = next((c for c in PAGE_COLUMNS if c in test_df.columns and (test_df[c].astype(str) != '').any()), None)
if page_column:
    train_pages = set(train_df[page_column].astype(str)) - {''}
    test_on_train_pages = int(test_df[page_column].astype(str).isin(train_pages).sum())
else:
    test_on_train_pages = None
real_test = 'source' not in test_df.columns or bool((test_df['source'].astype(str).str.lower() == 'real').any())

# Label lengths in tokens: training cuts longer labels at MAX_LABEL_LENGTH (it keeps the end token
# </s>, so the model learns to stop too early on them), and CRMS reads at most EVAL_MAX_NEW_TOKENS.
texts = sorted(set(train_df['label']) | set(val_df['label']) | set(test_df['label']))
token_len = dict(zip(texts, (len(ids) for ids in tokenizer(texts)['input_ids'])))
train_cut = int(sum(token_len[t] > cfg['MAX_LABEL_LENGTH'] for t in train_df['label']))
# token_len counts <s> + text + </s>; a reading is whole when <s> + text fits in EVAL_MAX_NEW_TOKENS.
test_too_long = int(sum(token_len[t] - 1 > EVAL_MAX_NEW_TOKENS for t in test_df['label']))

# Picture shapes, with the CRMS border: CRMS line crops are about 2.6x wider than tall (middle value; most fall
# between 0.9x and 3.9x). TrOCR squeezes every picture to 384x384, so a very different shape
# here than in CRMS means the model trains on differently squeezed letters.
shape_sample = train_df['path'].sample(n=min(500, len(train_df)), random_state=SEED)
sizes = []
for path in shape_sample:
    with Image.open(path) as opened:
        sizes.append((opened.width + 2 * CRMS_BORDER_PX, opened.height + 2 * CRMS_BORDER_PX))
aspects = sorted(w / max(h, 1) for w, h in sizes)
heights = sorted(h for _, h in sizes)

data_summary = {
    'usable_rows': n_samples,
    'train_samples': len(train_df),
    'train_copies_of_test_removed': len(train_all) - len(train_unique),
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'val_samples_scored_per_epoch': len(val_df),
    'val_data_root': VAL_DATA_ROOT or None,
    'val_copies_removed': len(val_all) - len(val_usable),
    'test_samples': len(test_df),
    'test_texts_seen_in_training': int(test_df['label_seen'].sum()),
    'duplicate_check': CHECK_DUPLICATE_IMAGES,
    'page_column': page_column,
    'test_images_from_training_pages': test_on_train_pages,
    'crms_border_px': CRMS_BORDER_PX,
    'longest_label_tokens': max(token_len.values()),
    'training_labels_cut_at_max_label_length': train_cut,
    'test_labels_longer_than_crms_reads': test_too_long,
    'train_image_aspect_w_over_h': {'p10': round(aspects[len(aspects) // 10], 2),
                                    'median': round(aspects[len(aspects) // 2], 2),
                                    'p90': round(aspects[len(aspects) * 9 // 10], 2)},
    'train_image_height_px_median': heights[len(heights) // 2],
}

print(f"Training   : {len(train_df):,} images"
      + (f" ({data_summary['train_copies_of_test_removed']:,} exact copies of test images left out)"
         if data_summary['train_copies_of_test_removed'] else ''))
print(f"Validation : {len(val_df):,} images scored after each epoch"
      + (f' from {os.path.basename(VAL_DATA_ROOT)}' if VAL_DATA_ROOT else '')
      + (f" ({data_summary['val_copies_removed']:,} copies of test/training images left out)"
         if data_summary['val_copies_removed'] else ''))
print(f"Test       : {len(test_df):,} images (locked: never trained on, scored once before and once after)")
seen, unseen = data_summary['test_texts_seen_in_training'], len(test_df) - data_summary['test_texts_seen_in_training']
print(f"             {seen:,} test texts also appear in training, {unseen:,} do not")
if unseen == 0:
    print('NOTE: every test text also appears in training. The test then checks reading new pictures of '
          'known texts, not new texts. Say so when you report the scores.')
if page_column is None and real_test:
    print(f"NOTE: manifest.csv has no page column ({', '.join(PAGE_COLUMNS)}), so the split cannot be checked "
          'by page. If it was split crop by crop, one page can have crops in training and in test, and the test '
          'scores are better than a new ledger book will get.')
elif test_on_train_pages:
    print(f'WARNING: {test_on_train_pages:,} test images come from pages that also have crops in training. '
          'Split the dataset by page for a fair test.')
print(f"Longest label: {data_summary['longest_label_tokens']} tokens")
if train_cut:
    print(f"WARNING: {train_cut:,} training labels are longer than MAX_LABEL_LENGTH "
          f"({cfg['MAX_LABEL_LENGTH']} tokens) and get cut. Raise MAX_LABEL_LENGTH in cell 4.")
if test_too_long:
    print(f'WARNING: {test_too_long:,} test labels are longer than CRMS reads ({EVAL_MAX_NEW_TOKENS} tokens). '
          'They count as wrong here, as they would in CRMS.')
a = data_summary['train_image_aspect_w_over_h']
print(f"Picture shape (width / height): {a['p10']} - {a['median']} - {a['p90']} (10% / middle / 90%), "
      f"middle height {data_summary['train_image_height_px_median']} px. CRMS crops: about 0.9 - 2.6 - 3.9.")

In [ ]:
# 8. CRMS border (every picture), picture variation (training only), dataset and loader
def crms_pad(image, px=None):
    """The border CRMS puts around every crop (ml/line_markers.py crop_line): `px` pixels
    (CRMS_BORDER_PX) of the paper's own colour, the 75th percentile of the crop's pixels."""
    px = CRMS_BORDER_PX if px is None else px
    if px <= 0:
        return image
    fill = tuple(int(v) for v in np.percentile(np.asarray(image).reshape(-1, 3), 75, axis=0))
    return ImageOps.expand(image, border=px, fill=fill)


def border_colour(image):
    """The paper colour, from the four corners, to fill new margins with."""
    w, h = image.size
    corners = [image.getpixel(p) for p in ((0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1))]
    return tuple(int(np.median([c[i] for c in corners])) for i in range(3))


def vary(image, rng=random):
    """Small random changes that real scans have and clean synthetic pictures may not.
    Each one is applied to only some pictures; the text itself is never cut off."""
    fill = border_colour(image)
    if rng.random() < 0.3:   # narrower or wider handwriting
        w, h = image.size
        image = image.resize((max(8, round(w * rng.uniform(0.8, 1.2))), h), Image.BICUBIC)
    if rng.random() < 0.4:   # a slightly tilted line
        image = image.rotate(rng.uniform(-2.5, 2.5), resample=Image.BICUBIC, expand=True, fillcolor=fill)
    if rng.random() < 0.3:   # more margin around the text
        w, h = image.size
        image = ImageOps.expand(image, fill=fill, border=(
            round(w * rng.uniform(0, 0.08)), round(h * rng.uniform(0, 0.15)),
            round(w * rng.uniform(0, 0.08)), round(h * rng.uniform(0, 0.15))))
    if min(image.size) >= 32 and rng.random() < 0.2:   # thicker or thinner ink
        image = image.filter(ImageFilter.MinFilter(3) if rng.random() < 0.5 else ImageFilter.MaxFilter(3))
    if rng.random() < 0.3:   # lighter or darker scan
        image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.7, 1.3))
        image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.85, 1.15))
    if rng.random() < 0.2:   # slightly out of focus
        image = image.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.0)))
    if rng.random() < 0.15:  # scanner compression
        buffer = io.BytesIO()
        image.save(buffer, 'JPEG', quality=rng.randint(40, 85))
        buffer.seek(0)
        image = Image.open(buffer).convert('RGB')
    return image


class HandwrittenDataset(Dataset):
    def __init__(self, frame, processor, max_len, augment=False):
        self.paths = frame['path'].tolist()
        self.labels = frame['label'].tolist()
        self.processor = processor
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as opened:
            image = crms_pad(opened.convert('RGB'))
        if self.augment:
            image = vary(image)
        pixel_values = self.processor(images=image, return_tensors='pt').pixel_values.squeeze(0)
        labels = self.processor.tokenizer(self.labels[idx], padding='max_length', max_length=self.max_len,
                                          truncation=True, return_tensors='pt').input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}


train_ds = HandwrittenDataset(train_df, processor, cfg['MAX_LABEL_LENGTH'], augment=AUGMENT)
# persistent_workers keeps the loader workers between epochs; without it the old ones are shut
# down from the wrong process and print a harmless red "can only test a child process" error.
train_loader = DataLoader(train_ds, batch_size=cfg['BATCH_SIZE'], shuffle=True,
                          num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
                          pin_memory=True, drop_last=True,
                          generator=torch.Generator().manual_seed(SEED))
if len(train_loader) == 0:
    raise ValueError('Fewer training samples than one batch.')
print('Training batches per epoch:', len(train_loader))

# A preview to check by eye that the varied pictures are still readable: original left, varied right.
if AUGMENT:
    preview_rng = random.Random(SEED)
    rows = []
    for path in train_df['path'].sample(n=min(8, len(train_df)), random_state=SEED):
        with Image.open(path) as opened:
            original = crms_pad(opened.convert('RGB'))
        pair = [original, vary(original, preview_rng)]
        pair = [p.resize((max(1, round(p.width * 64 / p.height)), 64)) for p in pair]
        rows.append(pair)
    width = max(a.width + b.width + 30 for a, b in rows)
    sheet = Image.new('RGB', (width, 74 * len(rows) + 10), 'white')
    for i, (a, b) in enumerate(rows):
        sheet.paste(a, (10, 10 + 74 * i))
        sheet.paste(b, (20 + a.width, 10 + 74 * i))
    preview_path = os.path.join(WORKING_DIR, 'augmentation-preview.png')
    sheet.save(preview_path)
    print('Preview of varied training pictures:', preview_path)
    try:
        from IPython.display import display
        display(sheet)
    except ImportError:
        pass

In [ ]:
# 9. Reading and scoring - the way CRMS reads: greedy, its border, at most EVAL_MAX_NEW_TOKENS tokens
def open_rgb(path):
    with Image.open(path) as opened:
        return crms_pad(opened.convert('RGB'))  # the same border as in training and in CRMS


image_pool = ThreadPoolExecutor(max_workers=8)  # opens the next pictures while the GPU reads


@torch.no_grad()
def read_images(paths, model, desc):
    model.eval()
    texts, seconds = [], 0.0
    batches = [paths[i:i + EVAL_BATCH_SIZE] for i in range(0, len(paths), EVAL_BATCH_SIZE)]
    pending = image_pool.submit(lambda b: list(map(open_rgb, b)), batches[0]) if batches else None
    for i in tqdm(range(len(batches)), desc=desc, leave=False):
        images = pending.result()
        if i + 1 < len(batches):
            pending = image_pool.submit(lambda b: list(map(open_rgb, b)), batches[i + 1])
        # In the model's own precision: fp16 for a test copy from load_for_test, as CRMS feeds it.
        pixel_values = processor(images=images, return_tensors='pt').pixel_values.to(device, dtype=model.dtype)
        torch.cuda.synchronize()
        started = time.perf_counter()
        generated = model.generate(pixel_values, max_new_tokens=EVAL_MAX_NEW_TOKENS, use_cache=True)
        torch.cuda.synchronize()
        seconds += time.perf_counter() - started
        texts.extend(text.strip() for text in processor.batch_decode(generated, skip_special_tokens=True))
    return texts, seconds


def score(frame, model, title=None):
    """Metrics plus each image's reading."""
    predictions, seconds = read_images(frame['path'].tolist(), model, title or 'Reading')
    result = compute_metrics(frame['label'].tolist(), predictions)
    result['ms_per_image'] = seconds / max(len(frame), 1) * 1000
    result['predictions'] = predictions
    result['correct'] = [as_crms(p) == as_crms(r) for p, r in zip(predictions, frame['label'])]
    if title:
        print_metrics(result, title)
    return result


def score_subset(frame, result, mask):
    refs = [r for r, keep in zip(frame['label'], mask) if keep]
    hyps = [h for h, keep in zip(result['predictions'], mask) if keep]
    return compute_metrics(refs, hyps)


def public(result):
    """Only the numbers, for the reports."""
    if result is None:
        return None
    return {key: result[key] for key in ('cer', 'wer', 'accuracy', 'exact', 'total', 'ms_per_image') if key in result}


print('[OK] Evaluator')

In [ ]:
# 10. BEFORE: Microsoft's base model on the locked test set, and on validation (epoch 0)
# The test is read by a copy kept the way CRMS reads (READ_PRECISION), for base-evaluation-report.json;
# validation by raw_model - still the unmodified base here - in fp32, like every later epoch.
# The decoder-start fix only re-affirms the model's own convention (id 2), so this is true base performance.
results = {}
test_reader = load_for_test(MODEL_NAME, revision=MODEL_REVISION) if READ_DTYPE != torch.float32 else raw_model
results['base'] = score(test_df, test_reader, f'BASE MODEL - TEST (before fine-tuning, {READ_PRECISION})')
del test_reader
gc.collect()
torch.cuda.empty_cache()
base_val = score(val_df, raw_model)
print(f"Base model on validation: CER {base_val['cer']*100:.2f}%")
# The pinned weights scored above. CRMS checks this hash against its own ml/models/base.
base_weights_sha256 = sha256_file(hf_hub_download(MODEL_NAME, 'model.safetensors', revision=MODEL_REVISION))

In [ ]:
# 11. Optimizer (separate encoder/decoder learning rates), encoder freezing, warmup + linear decay, both GPUs
def parameter_groups(model):
    """Decoder and encoder at their own learning rates. Biases and layer norms get no weight
    decay (the usual practice: shrinking them towards zero does not help)."""
    groups = []
    for part, lr in (('decoder', cfg['DECODER_LR']), ('encoder', cfg['ENCODER_LR'])):
        decay, plain = [], []
        for name, param in model.named_parameters():
            if name.startswith('encoder.') != (part == 'encoder'):
                continue
            (plain if name.endswith('bias') or 'norm' in name.lower() else decay).append(param)
        groups += [{'params': decay, 'lr': lr, 'weight_decay': cfg['WEIGHT_DECAY']},
                   {'params': plain, 'lr': lr, 'weight_decay': 0.0}]
    return groups


if cfg['OPTIMIZER'] == 'adafactor':
    # Fits a 6 GB GPU: a small running summary per weight instead of AdamW's two full copies. The learning
    # rates are the profile's fixed ones (no relative step); warmup and decay come from the scheduler below.
    optimizer = Adafactor(parameter_groups(raw_model), lr=cfg['DECODER_LR'], scale_parameter=False,
                          relative_step=False, warmup_init=False)
else:
    optimizer = AdamW(parameter_groups(raw_model))
if cfg['GRADIENT_CHECKPOINTING']:
    # Keeps a 6 GB GPU inside its own memory (see GPU_SETUPS in cell 4). Only training recomputes;
    # reading (generate) works as before.
    raw_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

if cfg['FREEZE_ENCODER']:
    for p in raw_model.encoder.parameters():
        p.requires_grad = False
    print('Encoder FROZEN (unfreezes at epoch', cfg['UNFREEZE_AT_EPOCH'] + 1, ')')

updates_per_epoch = math.ceil(len(train_loader) / cfg['GRAD_ACCUM_STEPS'])
total_updates = updates_per_epoch * cfg['EPOCHS']
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_updates * cfg['WARMUP_RATIO']), total_updates)

# Both T4s for the training forward pass. Reading stays on raw_model (1 GPU).
model = raw_model
if n_gpu > 1:
    model = torch.nn.DataParallel(raw_model)
    print(f'Using DataParallel across {n_gpu} GPUs')
scaler = GradScaler('cuda')
print(f"[OK] Optimizer {cfg['OPTIMIZER']} | {updates_per_epoch} updates per epoch, {total_updates} in all")

In [ ]:
# 12. TRAINING - mixed precision, gradient accumulation, scheduled unfreeze, best epoch by validation CER, early stop
if os.path.isdir(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)  # a re-run must never package files from an earlier run
os.makedirs(OUTPUT_DIR)

epoch_log = [{'epoch': 0, 'train_loss': None, 'val_cer': base_val['cer'], 'val_exact': base_val['accuracy'],
              'minutes': 0.0}]
# An epoch is kept only when it reads validation better than the base model (epoch 0).
best_cer, best_epoch, patience = base_val['cer'], None, 0
started = time.time()

for epoch in range(1, cfg['EPOCHS'] + 1):
    epoch_started = time.time()
    if cfg['FREEZE_ENCODER'] and epoch == cfg['UNFREEZE_AT_EPOCH'] + 1:
        for p in raw_model.encoder.parameters():
            p.requires_grad = True
        print(f'  -> Encoder UNFROZEN at epoch {epoch}')

    model.train()
    running = 0.0
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg["EPOCHS"]}')
    for step, batch in enumerate(progress, start=1):
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        with autocast('cuda'):
            loss = model(pixel_values=pixel_values, labels=labels).loss.mean() / cfg['GRAD_ACCUM_STEPS']
        scaler.scale(loss).backward()
        running += loss.item() * cfg['GRAD_ACCUM_STEPS']
        # Step every GRAD_ACCUM_STEPS batches, and on the last batch so no gradient is thrown away.
        if step % cfg['GRAD_ACCUM_STEPS'] == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)  # unscale before clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        progress.set_postfix(loss=f'{running/step:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    val = score(val_df, raw_model)
    summary = {'epoch': epoch, 'train_loss': running / len(train_loader), 'val_cer': val['cer'],
               'val_exact': val['accuracy'], 'minutes': round((time.time() - epoch_started) / 60, 1)}
    epoch_log.append(summary)
    print(f"  Epoch {epoch}: train_loss={summary['train_loss']:.4f}  val_CER={val['cer']*100:.2f}%  "
          f"val_exact={val['accuracy']*100:.2f}%  ({summary['minutes']} min)")
    if val['cer'] < best_cer:
        best_cer, best_epoch, patience = val['cer'], epoch, 0
        raw_model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        print(f'    saved as the best epoch so far (val_CER={best_cer*100:.2f}%)')
    else:
        patience += 1
        print(f"    no improvement ({patience}/{cfg['EARLY_STOP_PATIENCE']})")
        if patience >= cfg['EARLY_STOP_PATIENCE']:
            print('    early stopping')
            break

training_minutes = (time.time() - started) / 60
print(f'\nDone in {training_minutes:.1f} min. Best epoch: {best_epoch} (val CER {best_cer*100:.2f}%); '
      f"the base model had {base_val['cer']*100:.2f}%.")

In [ ]:
# 13. AFTER: the best epoch on the same locked test set
for name in ('model', 'optimizer', 'scheduler', 'scaler', 'raw_model', 'batch', 'pixel_values', 'labels', 'loss'):
    globals().pop(name, None)  # drop every reference to the training model, to free the GPU
gc.collect()
torch.cuda.empty_cache()
if best_epoch is None:
    raise RuntimeError(f"No epoch read validation better than the base model (CER {base_val['cer']*100:.2f}%), "
                       'so nothing was saved and there is nothing to upload.')
best_model = load_for_test(OUTPUT_DIR)
results['finetuned'] = score(test_df, best_model, f'FINE-TUNED MODEL ({CRMS_MODEL_KEY}) - TEST ({READ_PRECISION})')
print('Model saved at:', OUTPUT_DIR)

In [ ]:
# 14. SIDE-BY-SIDE COMPARISON on the same test images (CER/WER: lower is better | exact match: higher is better)
def pct(value):
    return '-' if value is None else f'{value*100:.2f}%'


base, tuned = results['base'], results['finetuned']
print('=' * 66)
print(f'LOCKED TEST - {DATASET_LABEL} ({len(test_df):,} images)')
print('-' * 66)
print(f"{'':16}{'Base':>14}{'Fine-tuned':>16}{'Verdict':>14}")
for key, label, lower_is_better in (('cer', 'CER', True), ('wer', 'WER', True), ('accuracy', 'Exact match', False)):
    b, f = base[key], tuned[key]
    verdict = 'same' if f == b else ('better' if (f < b) == lower_is_better else 'worse')
    print(f'{label:16}{pct(b):>14}{pct(f):>16}{verdict:>14}')

seen_mask = test_df['label_seen'].tolist()
unseen_mask = [not s for s in seen_mask]
subsets = {}
for name, mask in (('texts_seen_in_training', seen_mask), ('texts_not_in_training', unseen_mask)):
    if any(mask):
        subsets[name] = {'images': sum(mask), 'base': score_subset(test_df, base, mask),
                         'finetuned': score_subset(test_df, tuned, mask)}
for name, label in (('texts_seen_in_training', 'Seen texts'), ('texts_not_in_training', 'Unseen texts')):
    if name in subsets:
        s = subsets[name]
        print(f"{label:16}{pct(s['base']['accuracy']):>14}{pct(s['finetuned']['accuracy']):>16}"
              f"   exact match on {s['images']:,} images")
print('-' * 66)
fixed = sum(1 for b, f in zip(base['correct'], tuned['correct']) if f and not b)
broke = sum(1 for b, f in zip(base['correct'], tuned['correct']) if b and not f)
p_value = mcnemar_p(fixed, broke)
print(f'Fine-tuned vs base: fixed {fixed:,} images, broke {broke:,}. '
      f'Chance check p = {p_value:.3g} ({"unlikely to be chance" if p_value < 0.05 else "could be chance"})')
print(f"Reading time: base {base['ms_per_image']:.1f} ms, fine-tuned {tuned['ms_per_image']:.1f} ms per image "
      f'({torch.cuda.get_device_name(0)}, batches of {EVAL_BATCH_SIZE}, {READ_PRECISION})')
print('=' * 66)
if 'texts_not_in_training' not in subsets:
    print('Every test text also appears in training: these scores show reading new pictures of known texts.')
print('All scores are on this dataset\'s test pictures; crops cut by CRMS itself can score differently.')

In [ ]:
# 15. Reports for CRMS, bound to the exact weights they were measured on
# Two reports, both from the locked TEST set of this one run:
#   evaluation-report.json       - the fine-tuned model (cell 13)
#   base-evaluation-report.json  - the base model before fine-tuning (cell 10)
# Each names its weights by SHA-256. CRMS checks the fine-tuned report against the uploaded
# weights, and the base report against the base model installed in CRMS; the base report is
# installed only if both match. CRMS reads the standard fields and ignores the extra blocks.
weights_file = weights_file_in(OUTPUT_DIR)
if weights_file is None:
    raise FileNotFoundError('The best epoch was not saved.')
assert base['total'] == tuned['total'], 'Base and fine-tuned were not scored on the same test set.'
manifest_sha256 = sha256_file(MANIFEST_CSV)
evaluated_at = datetime.now(timezone.utc).isoformat()
versions = {'torch': torch.__version__, 'transformers': transformers.__version__}


def crms_report(model_key, metrics, weights_name, weights_sha256):
    return {
        'schema_version': 1,
        'model_key': model_key,
        'base_model': MODEL_NAME,
        'base_model_revision': MODEL_REVISION,
        'dataset': DATASET_LABEL,
        'manifest_sha256': manifest_sha256,
        'split': 'test',
        'sample_count': int(metrics['total']),
        'metrics': {
            'cer': float(metrics['cer']),
            'wer': float(metrics['wer']),
            'exact_match': float(metrics['accuracy']),
        },
        'evaluated_at': evaluated_at,
        'weights_file': weights_name,
        'weights_sha256': weights_sha256,
    }


report = crms_report(CRMS_MODEL_KEY, tuned, weights_file, sha256_file(os.path.join(OUTPUT_DIR, weights_file)))
report['reading'] = {'max_new_tokens': EVAL_MAX_NEW_TOKENS, 'decoding': 'greedy', 'precision': READ_PRECISION,
                     'border_px': CRMS_BORDER_PX,
                     'ms_per_image': round(tuned['ms_per_image'], 1), 'device': torch.cuda.get_device_name(0)}
report['training'] = {
    'kind': 'from_base', 'profile': PROFILE, 'hyperparameters': cfg, 'seed': SEED, 'augment': AUGMENT,
    'train_samples': len(train_df), 'epochs_run': len(epoch_log) - 1, 'best_epoch': best_epoch,
    'best_val_cer': best_cer, 'training_minutes': round(training_minutes, 1), 'versions': versions,
    'run_on': 'kaggle' if IS_KAGGLE else 'local', 'gpu_setup': GPU_SETUP,
    'gpu': f'{torch.cuda.get_device_name(0)} x{max(n_gpu, 1)}',
}
report['test_subsets'] = {name: {'images': s['images'], 'metrics': s['finetuned']} for name, s in subsets.items()}
base_report = crms_report('base', base, 'model.safetensors', base_weights_sha256)
base_report['reading'] = {**report['reading'], 'ms_per_image': round(base['ms_per_image'], 1)}

summary = {
    'schema': 'crms-training-summary/1',
    'created_at': evaluated_at,
    'model': {'key': CRMS_MODEL_KEY, 'weights_sha256': report['weights_sha256'], 'best_epoch': best_epoch},
    'base_model': {'name': MODEL_NAME, 'revision': MODEL_REVISION, 'weights_sha256': base_weights_sha256},
    'dataset': {'label': DATASET_LABEL, 'root': DATA_ROOT, 'manifest_sha256': manifest_sha256,
                'validation_root': VAL_DATA_ROOT or DATA_ROOT},
    'data': data_summary,
    'settings': {'PROFILE': PROFILE, **cfg, 'RUN_ON': 'kaggle' if IS_KAGGLE else 'local', 'GPU_SETUP': GPU_SETUP,
                 'MAX_TRAIN_SAMPLES': MAX_TRAIN_SAMPLES, 'EVAL_MAX_NEW_TOKENS': EVAL_MAX_NEW_TOKENS,
                 'VAL_SAMPLES': VAL_SAMPLES,
                 'SEED': SEED, 'AUGMENT': AUGMENT, 'CHECK_DUPLICATE_IMAGES': CHECK_DUPLICATE_IMAGES,
                 'CRMS_BORDER_PX': CRMS_BORDER_PX, 'READ_PRECISION': READ_PRECISION, 'VAL_DATA_ROOT': VAL_DATA_ROOT},
    'scores': {'base': public(base), 'finetuned': public(tuned)},
    'test_subsets': {name: {'images': s['images'], 'base': s['base'], 'finetuned': s['finetuned']}
                     for name, s in subsets.items()},
    'finetuned_vs_base': {'fixed': fixed, 'broke': broke, 'mcnemar_p': p_value},
    'epoch_log': epoch_log,
    'training_minutes': round(training_minutes, 1),
    'versions': versions,
}

for name, content in (('evaluation-report.json', report), ('base-evaluation-report.json', base_report),
                      ('training-summary.json', summary)):
    with open(os.path.join(OUTPUT_DIR, name), 'w', encoding='utf-8') as output:
        json.dump(content, output, indent=2)
shutil.copy2(os.path.join(OUTPUT_DIR, 'training-summary.json'), os.path.join(WORKING_DIR, 'training-summary.json'))

# Every test image either model read wrong, with both readings.
rows = [{'filename': test_df['filename'][i], 'label': test_df['label'][i],
         'base': base['predictions'][i], 'finetuned': tuned['predictions'][i],
         'text_seen_in_training': bool(test_df['label_seen'][i])}
        for i in range(len(test_df)) if not (base['correct'][i] and tuned['correct'][i])]
errors_csv = os.path.join(WORKING_DIR, 'test-errors.csv')
pd.DataFrame(rows, columns=['filename', 'label', 'base', 'finetuned', 'text_seen_in_training']).to_csv(
    errors_csv, index=False)

print('CRMS report (fine-tuned model, locked test):')
print(json.dumps(report, indent=2))
print('Base report: exact', f"{base_report['metrics']['exact_match']*100:.2f}%", 'on the same',
      base_report['sample_count'], 'images')
print('Errors list:', errors_csv, f'({len(rows):,} rows)')

In [ ]:
# 16. Package the checkpoint and its reports for CRMS
archive_path = OUTPUT_DIR.rstrip('/') + '.zip'
if os.path.exists(archive_path):
    os.remove(archive_path)
archive = shutil.make_archive(OUTPUT_DIR.rstrip('/'), 'zip', OUTPUT_DIR)
with zipfile.ZipFile(archive) as packed_zip:
    packed = set(packed_zip.namelist())
for required in ('config.json', 'generation_config.json', weights_file, 'evaluation-report.json',
                 'base-evaluation-report.json', 'training-summary.json'):
    assert required in packed, f'{required} is missing from the ZIP - rerun cells 12 to 15.'
print('zipped ->', archive, f'({os.path.getsize(archive) / 1e9:.2f} GB)')
print('Upload it in the CRMS OCR Workspace as', repr(CRMS_MODEL_KEY))
if not IS_KAGGLE:
    print('Run folder:', WORKING_DIR)